# Cardiac aging MSI: first inspection

Twelve independent mice, six young and six old. This notebook first inspects all
file headers, then reads a small, reproducible sample of spectra from one mouse.
It never loads the complete intensity matrix into RAM.

Run from the project environment after following README.md. Data stay on F:.
The final spatial preview streams the selected mouse's XML and reads one narrow
mass window per pixel. It can take several minutes on an external drive.


In [ ]:
from pathlib import Path
import sys
import json
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository.'
sys.path.insert(0, str(ROOT / 'src'))
from cardiac_aging.inspection import load_config, inventory
config = load_config(ROOT)
DATA = Path(config['data_dir'])
RESULTS = ROOT / config['results_dir']
RESULTS.mkdir(parents=True, exist_ok=True)


## File inventory and biological replication

Missing metadata remains blank. Filename dates are retained without interpreting them as acquisition dates. Group comparisons later must use mice as independent replicates.

In [ ]:
rows = inventory(DATA, ROOT / config['sample_sheet'])
table = pd.DataFrame(rows)
display(table[['measurement_id', 'original_sample_id', 'age_group', 'spectra',
               'pixel_x', 'pixel_y', 'mz_min', 'mz_max', 'size_consistent']])
display(table.groupby('age_group')['animal_id'].nunique().rename('independent_mice'))
assert table['ibd_exists'].all(), 'A paired ibd file is missing.'
assert not (table['size_consistent'] == False).any(), 'Unexpected binary file size.'
(RESULTS / 'inventory.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')


## Selected spectra from mouse 1465

The supplied files declare a continuous, uncompressed shared mass axis. The following small reader checks that layout before using memory maps. Unsupported layouts stop with an error. The per-pixel signal sums help assess existing TIC normalization; they do not recover original unnormalized signals.

In [ ]:
selected = table.loc[table['measurement_id'] == '1465'].iloc[0]
path = DATA / selected['imzml_filename']
assert selected['size_consistent'] is True or selected['size_consistent'] == True
points, pixels = int(selected['mz_points']), int(selected['spectra'])
ibd = path.with_suffix('.ibd')
mz = np.memmap(ibd, dtype='<f8', mode='r', offset=16, shape=(points,))
intensities = np.memmap(ibd, dtype='<f4', mode='r', offset=16 + points * 8,
                        shape=(pixels, points))
assert np.isfinite(mz).all() and (np.diff(mz) > 0).all()
rng = np.random.default_rng(config['random_seed'])
indices = np.sort(rng.choice(pixels, size=min(config['spectra_per_mouse'], pixels), replace=False))
sampled = np.asarray(intensities[indices], dtype=np.float64)
assert np.isfinite(sampled).all(), 'Non-finite sampled intensities.'
tic = sampled.sum(axis=1)
display(pd.DataFrame({'pixel_index': indices, 'signal_sum': tic}))
print(f'Sampled {len(indices)} of {pixels:,} pixels; relative SD of signal sums: {tic.std()/tic.mean():.3g}')
print('Near-constant sums support prior TIC normalization. Confirm the export settings before normalizing again.')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
axes[0].plot(mz, sampled.mean(axis=0), linewidth=0.6)
axes[0].set(title='Mouse 1465: mean of sampled spectra', xlabel='m/z', ylabel='Exported intensity')
window = (mz >= 1528) & (mz <= 1535)
axes[1].plot(mz[window], sampled[:, window].mean(axis=0), linewidth=1)
axes[1].axvline(1531.785, color='tab:red', linestyle='--', label='Slide ion-map centre')
axes[1].set(xlabel='m/z', ylabel='Exported intensity', title='Candidate region near m/z 1531.8')
axes[1].legend()
fig.savefig(RESULTS / '1465_sampled_spectra.png', dpi=160)
plt.show()


## Spatial preview of the candidate ion

This sums signal within 1531.785 ± 250 ppm, matching the mass window labelled in the slide. It does not identify a molecule or establish an age effect. Only mouse 1465 is shown. Empty image positions remain NaN. Ion-window signal is a relative measure from the exported spectra.

In [ ]:
centre, ppm = 1531.785, 250
lo, hi = np.searchsorted(mz, [centre * (1 - ppm * 1e-6), centre * (1 + ppm * 1e-6)])
xs, ys, signal = [], [], []
def params(element):
    return {e.get('name'): e.get('value') for e in element.iter() if e.tag.endswith('cvParam')}

with path.open('rb') as handle, ibd.open('rb') as binary:
    for _, element in ET.iterparse(handle, events=('end',)):
        if element.tag.rsplit('}', 1)[-1] != 'spectrum':
            continue
        p = params(element)
        arrays = [e for e in element.iter() if e.tag.rsplit('}', 1)[-1] == 'binaryDataArray']
        intensity_array = next(a for a in arrays if any(e.get('ref') == 'intensities' for e in a))
        a = params(intensity_array)
        binary.seek(int(a['external offset']) + int(lo) * 4)
        values = np.frombuffer(binary.read(int(hi - lo) * 4), dtype='<f4')
        if len(values) != hi - lo:
            raise EOFError('Incomplete intensity data')
        xs.append(int(p['position x']))
        ys.append(int(p['position y']))
        signal.append(float(values.sum(dtype=np.float64)))
        element.clear()
assert len(signal) == pixels
image = np.full((max(ys), max(xs)), np.nan)
image[np.array(ys) - 1, np.array(xs) - 1] = signal
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
plot = ax.imshow(image, origin='upper', cmap='viridis',
                 vmin=np.nanpercentile(image, 1), vmax=np.nanpercentile(image, 99))
ax.set(title=f'Mouse 1465: m/z {centre} ± {ppm} ppm', xlabel='Pixel x', ylabel='Pixel y')
fig.colorbar(plot, ax=ax, label='Summed exported intensity (1–99% display limits)')
fig.savefig(RESULTS / '1465_candidate_ion.png', dpi=160)
plt.show()


## Next analysis

Review tissue masks, comparable anatomy and signal quality across all mice.
Confirm normalization with the operator. Build a compact peak-feature matrix
before group comparisons. Keep effect sizes and uncertainty at mouse level.
Any later ML split must keep every pixel from a mouse together, and feature
selection must happen within training folds. With twelve mice, prediction is
exploratory and requires an independent cohort for stronger validation.
